# Classifying Handwritten Digits with CNNs vs Fully Connected Networks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

print('TensorFlow :', tf.__version__)
print('Keras      :', keras.__version__)

## 1. Load the MNIST Dataset

In [ ]:
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.mnist.load_data()

print('=== Raw dataset shapes ===')
print(f'  X_train : {X_train_raw.shape}  — {X_train_raw.shape[0]:,} images')
print(f'  y_train : {y_train_raw.shape}')
print(f'  X_test  : {X_test_raw.shape}   — {X_test_raw.shape[0]:,} images')
print(f'  y_test  : {y_test_raw.shape}')
print(f'  Pixel range : {X_train_raw.min()} – {X_train_raw.max()}')
print(f'  Classes     : {np.unique(y_train_raw)}')

In [ ]:
# ── Display sample images ─────────────────────────────────────
fig, axes = plt.subplots(3, 10, figsize=(16, 5))
np.random.seed(42)
for digit in range(10):
    idxs = np.where(y_train_raw == digit)[0]
    chosen = np.random.choice(idxs, 3, replace=False)
    for row, idx in enumerate(chosen):
        axes[row, digit].imshow(X_train_raw[idx], cmap='gray')
        axes[row, digit].axis('off')
        if row == 0:
            axes[row, digit].set_title(str(digit), fontsize=12,
                                       fontweight='bold', color='steelblue')

plt.suptitle('Sample Images — 3 examples per digit', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Preprocess for the Fully Connected Neural Network

In [ ]:
# Flatten 28×28 → 784, normalise, one-hot encode
X_train_fc = X_train_raw.reshape(-1, 784) / 255.0
X_test_fc  = X_test_raw.reshape(-1, 784)  / 255.0

NUM_CLASSES   = 10
y_train_ohe   = to_categorical(y_train_raw, NUM_CLASSES)
y_test_ohe    = to_categorical(y_test_raw,  NUM_CLASSES)

print('FCNN preprocessing complete.')
print(f'  X_train_fc shape : {X_train_fc.shape}  (flattened)')
print(f'  y_train_ohe shape: {y_train_ohe.shape} (one-hot)')
print(f'  Pixel range      : {X_train_fc.min():.1f} – {X_train_fc.max():.1f}')

## 3. Build and Train the Fully Connected Neural Network

In [ ]:
# ── Architecture ─────────────────────────────────────────────
fcnn = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(784,)),
    layers.Dropout(0.25),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.25),
    layers.Dense(128, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax'),
], name='Fully_Connected_NN')

fcnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
fcnn.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────────
EPOCHS     = 10
BATCH_SIZE = 128

history_fc = fcnn.fit(
    X_train_fc, y_train_ohe,
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    validation_split = 0.10,
    verbose          = 1
)

In [ ]:
fc_loss, fc_acc = fcnn.evaluate(X_test_fc, y_test_ohe, verbose=0)
print(f'FCNN — Test Accuracy : {fc_acc*100:.2f}%  |  Test Loss : {fc_loss:.4f}')

## 4. Preprocess for the Convolutional Neural Network

In [ ]:
# Reshape to (N, 28, 28, 1), normalise, one-hot encode
X_train_cnn = X_train_raw.reshape(-1, 28, 28, 1) / 255.0
X_test_cnn  = X_test_raw.reshape(-1, 28, 28, 1)  / 255.0

print('CNN preprocessing complete.')
print(f'  X_train_cnn shape : {X_train_cnn.shape}  (H × W × Channels)')
print(f'  Pixel range       : {X_train_cnn.min():.1f} – {X_train_cnn.max():.1f}')

## 5. Build and Train the Convolutional Neural Network

In [ ]:
# ── Architecture ─────────────────────────────────────────────
cnn = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                  input_shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPool2D((2, 2)),
    layers.Dropout(0.25),
    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPool2D((2, 2)),
    layers.Dropout(0.25),
    # Classifier head
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.40),
    layers.Dense(NUM_CLASSES, activation='softmax'),
], name='Convolutional_NN')

cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
cnn.summary()

In [ ]:
history_cnn = cnn.fit(
    X_train_cnn, y_train_ohe,
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    validation_split = 0.10,
    verbose          = 1
)

In [ ]:
cnn_loss, cnn_acc = cnn.evaluate(X_test_cnn, y_test_ohe, verbose=0)
print(f'CNN — Test Accuracy : {cnn_acc*100:.2f}%  |  Test Loss : {cnn_loss:.4f}')

## 6. Compare Performance

In [ ]:
print('='*52)
print(f'  {"Model":<25}  {"Test Acc":>10}  {"Test Loss":>10}')
print('-'*52)
print(f'  {"Fully Connected NN":<25}  {fc_acc*100:>9.2f}%  {fc_loss:>10.4f}')
print(f'  {"Convolutional NN":<25}  {cnn_acc*100:>9.2f}%  {cnn_loss:>10.4f}')
print('='*52)
print(f'  CNN gain : +{(cnn_acc - fc_acc)*100:.2f} percentage points')

In [ ]:
# ── Training history comparison ────────────────────────────────
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(epochs_range, history_fc.history['val_accuracy'],
             'o-', color='#4C72B0', linewidth=2, markersize=5, label='FCNN val')
axes[0].plot(epochs_range, history_cnn.history['val_accuracy'],
             's-', color='#E84040', linewidth=2, markersize=5, label='CNN val')
axes[0].plot(epochs_range, history_fc.history['accuracy'],
             '--', color='#4C72B0', linewidth=1, alpha=0.5, label='FCNN train')
axes[0].plot(epochs_range, history_cnn.history['accuracy'],
             '--', color='#E84040', linewidth=1, alpha=0.5, label='CNN train')
axes[0].set_title('Validation Accuracy — FCNN vs CNN', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_xticks(epochs_range)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(epochs_range, history_fc.history['val_loss'],
             'o-', color='#4C72B0', linewidth=2, markersize=5, label='FCNN val')
axes[1].plot(epochs_range, history_cnn.history['val_loss'],
             's-', color='#E84040', linewidth=2, markersize=5, label='CNN val')
axes[1].plot(epochs_range, history_fc.history['loss'],
             '--', color='#4C72B0', linewidth=1, alpha=0.5, label='FCNN train')
axes[1].plot(epochs_range, history_cnn.history['loss'],
             '--', color='#E84040', linewidth=1, alpha=0.5, label='CNN train')
axes[1].set_title('Loss — FCNN vs CNN', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Categorical Cross-Entropy')
axes[1].set_xticks(epochs_range)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training History Comparison: FCNN vs CNN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Side-by-side confusion matrices ────────────────────────────
y_pred_fc  = fcnn.predict(X_test_fc,  verbose=0).argmax(axis=1)
y_pred_cnn = cnn.predict( X_test_cnn, verbose=0).argmax(axis=1)

cm_fc  = confusion_matrix(y_test_raw, y_pred_fc)
cm_cnn = confusion_matrix(y_test_raw, y_pred_cnn)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, cm, title, acc in [
    (axes[0], cm_fc,  f'FCNN (Acc = {fc_acc*100:.2f}%)',  fc_acc),
    (axes[1], cm_cnn, f'CNN  (Acc = {cnn_acc*100:.2f}%)', cnn_acc),
]:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=range(10), yticklabels=range(10),
                linewidths=0.5, ax=ax)
    ax.set_title(f'Confusion Matrix — {title}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Confusion Matrices Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-class accuracy comparison ──────────────────────────────
per_fc  = cm_fc.diagonal()  / cm_fc.sum(axis=1)
per_cnn = cm_cnn.diagonal() / cm_cnn.sum(axis=1)

x = np.arange(10)
w = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, per_fc  * 100, w, label='FCNN', color='#4C72B0', edgecolor='white')
ax.bar(x + w/2, per_cnn * 100, w, label='CNN',  color='#E84040', edgecolor='white')
ax.axhline(fc_acc  * 100, color='#4C72B0', linestyle='--', linewidth=1.2, alpha=0.7)
ax.axhline(cnn_acc * 100, color='#E84040', linestyle='--', linewidth=1.2, alpha=0.7)
ax.set_title('Per-Digit Accuracy: FCNN vs CNN', fontsize=13, fontweight='bold')
ax.set_xlabel('Digit')
ax.set_ylabel('Accuracy (%)')
ax.set_xticks(x)
ax.set_ylim(85, 102)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Per-digit accuracy comparison:')
print(f'{'Digit':>6}  {'FCNN':>8}  {'CNN':>8}  {'Delta':>8}')
print('-'*35)
for d in range(10):
    delta = (per_cnn[d] - per_fc[d]) * 100
    sign  = '+' if delta >= 0 else ''
    print(f'{d:>6}  {per_fc[d]*100:>7.2f}%  {per_cnn[d]*100:>7.2f}%  {sign}{delta:>6.2f}%')

In [ ]:
# ── Architecture comparison table ─────────────────────────────
print('='*60)
print('  ARCHITECTURE COMPARISON')
print('='*60)
print(f'  {'Property':<30}  {'FCNN':>10}  {'CNN':>10}')
print('-'*60)
props = [
    ('Total Parameters',       f'{fcnn.count_params():,}',   f'{cnn.count_params():,}'),
    ('Spatial awareness',      'None',                        'Full (conv filters)'),
    ('Translation invariance', 'None',                        'Yes'),
    ('Input shape',            '(784,)',                       '(28,28,1)'),
    ('Test Accuracy',          f'{fc_acc*100:.2f}%',          f'{cnn_acc*100:.2f}%'),
    ('Test Loss',              f'{fc_loss:.4f}',              f'{cnn_loss:.4f}'),
]
for name, fc_val, cnn_val in props:
    print(f'  {name:<30}  {fc_val:>10}  {cnn_val:>10}')
print('='*60)

In [ ]:
# ── Visualise CNN feature maps (first conv layer filters) ──────
conv_layer = cnn.layers[0]   # first Conv2D
filters, _ = conv_layer.get_weights()
# filters shape: (3, 3, 1, 32) → 32 filters of 3x3x1
n_filters = filters.shape[-1]

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flatten()):
    if i < n_filters:
        f = filters[:, :, 0, i]
        f_norm = (f - f.min()) / (f.max() - f.min() + 1e-8)
        ax.imshow(f_norm, cmap='viridis')
        ax.set_title(f'F{i+1}', fontsize=7)
    ax.axis('off')

plt.suptitle('Learned Filters — First Conv2D Layer (32 filters, 3×3)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Show CNN errors vs FCNN errors ────────────────────────────
errors_fc  = set(np.where(y_pred_fc  != y_test_raw)[0])
errors_cnn = set(np.where(y_pred_cnn != y_test_raw)[0])
only_fc_wrong  = errors_fc  - errors_cnn   # fixed by CNN
only_cnn_wrong = errors_cnn - errors_fc    # broken by CNN

print(f'FCNN misclassified : {len(errors_fc):,}')
print(f'CNN  misclassified : {len(errors_cnn):,}')
print(f'Fixed by CNN       : {len(only_fc_wrong):,} images')
print(f'New errors by CNN  : {len(only_cnn_wrong):,} images')

# Show a few images the CNN fixed
if only_fc_wrong:
    sample_fixed = list(only_fc_wrong)[:12]
    fig, axes = plt.subplots(2, 6, figsize=(14, 5))
    for ax, idx in zip(axes.flatten(), sample_fixed):
        ax.imshow(X_test_raw[idx], cmap='gray')
        ax.set_title(f'True:{y_test_raw[idx]}\nFC:{y_pred_fc[idx]} CNN:{y_pred_cnn[idx]}',
                     fontsize=8, color='green', fontweight='bold')
        ax.axis('off')
    plt.suptitle('Images FCNN got wrong but CNN got right',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Analysis and Conclusions

### Why CNNs Outperform Fully Connected Networks on Image Data

| Aspect | Fully Connected NN | Convolutional NN |
|---|---|---|
| **Input representation** | Flattens 28×28 into 784 — loses all spatial structure | Preserves the 2D grid of pixels |
| **Spatial awareness** | None: pixel (0,0) and pixel (27,27) are treated as independent features | Conv filters detect local patterns (edges, curves) at every position |
| **Translation invariance** | None: a digit shifted by 1 pixel looks entirely different to the network | MaxPooling gives approximate translation invariance |
| **Parameter efficiency** | Large Dense layers need many weights to cover all pixel combinations | Weight sharing across the image: a 3×3 filter uses only 9 weights regardless of image size |
| **Hierarchical features** | Every neuron sees every pixel — no notion of locality | Early layers detect edges; later layers detect curves, loops, strokes — natural for digit recognition |

### Key Observations from the Experiment

- The **CNN achieves higher accuracy** on the test set despite often having fewer trainable parameters than the FCNN, because it uses those parameters more efficiently through weight sharing and local connectivity.
- The **confusion matrices** reveal that both models make similar classes of errors (e.g., confusing 4 and 9, or 3 and 8), but the CNN makes fewer total errors.
- The **learned filters** in the first Conv2D layer visualise oriented edge detectors and blob detectors — patterns the network discovered entirely from data, without any hand-engineering.
- The **training curves** show that the CNN reaches higher validation accuracy earlier and with less overfitting relative to its training accuracy, thanks to Dropout regularisation and the built-in inductive bias toward spatial structure.